# 07 · Household CPI shock and Cost-of-Living Distress Score

How much did the prices of each household's own basket rise between January 2024 (the price base of EICV7 real consumption) and September 2026 (the latest CPI month)? And which households are least able to absorb it?

| | |
|---|---|
| **Inputs** | `data/processed/household_master.parquet` (notebook 06), `data/processed/cpi_features.parquet` (notebook 03), `reports/data_inspection/expenditure_cpi_mapping.csv` |
| **Outputs** | `data/processed/household_cpi_shock.parquet`, `docs/expenditure_cpi_mapping.md`, `reports/figures/household_shock/*.png` |
| **Code** | `src/features/household_cpi_shock.py` |
| **Method** | `docs/household_cpi_shock_methodology.md` |

**Formula.** For household *h* in CPI area *a* (urban or rural):

price change_h = Σ_c s_hc × (I_c,a(Sep 2026) / I_c,a(Jan 2024) − 1)

- *s_hc* is the household's own EICV7 share of spending cell *c* (not the EICV4 CPI weights).
- *I* is the official NISR index assigned to the cell.
- Annualised inflation = (1 + price change)^(12/32) − 1.

**Rules.**
- Urban households use urban indices and rural households use rural indices.
- No CPI month after September 2026 is read, and no FinScope data is used.
- Estimates use the household weight `weight`; poverty rates use `pop_wt`.

In [ ]:
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "features" / "household_cpi_shock.py").exists())
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
from IPython.display import Image, Markdown, display

from src.features import household_cpi_shock as hs

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_rows", 80)
pd.set_option("display.float_format", "{:,.2f}".format)
FIG = hs.FIGURES_DIR
SEG = list(hs.DECOMPOSITION)

## 1. Prices: official index changes, January 2024 to September 2026

In [ ]:
cpi = hs.load_cpi()
master = pd.read_parquet(hs.MASTER_PATH)
changes = hs.price_changes(cpi)
print(f"CPI file covers {cpi['date'].min():%Y-%m} to {cpi['date'].max():%Y-%m}; only {hs.BASE_MONTH:%Y-%m} and {hs.END_MONTH:%Y-%m} are used ({hs.N_MONTHS} months).")
cc = hs.cell_changes(changes, "main").mul(100)
cc["urban index"] = [hs.INDEX_ASSIGNMENT[c][0] for c in cc.index]
cc["rural index"] = [hs.INDEX_ASSIGNMENT[c][1] for c in cc.index]
cc["in main basket"] = ["no" if c in hs.MAIN_EXCLUDED else "yes" for c in cc.index]
cc.rename(columns={"urban": "urban change %", "rural": "rural change %"})

In [ ]:
head = changes.loc["headline", "change"].mul(100)
print(f"Published headline CPI change over the same months: urban {head['urban']:.1f}%, rural {head['rural']:.1f}%, Rwanda {head['national']:.1f}%")
print("Rural fallback cells (no rural index of their own):", ", ".join(hs.FALLBACK_CELLS))
flagged = changes[changes["flagged_months"].ne("")].reset_index()[["category", "area", "flagged_months"]]
used = {(u, "urban") for u, _, _ in hs.INDEX_ASSIGNMENT.values()} | {(r, "rural") for _, r, _ in hs.INDEX_ASSIGNMENT.values()}
flagged[[ (c, a) in used for c, a in zip(flagged["category"], flagged["area"])]]

These are the official index values, used unchanged. Flagged months are abrupt moves found in notebook 03; the largest are rural communication (+38% in October 2024, +55% in August 2026) and the September school-year jumps in education. Section 7 tests how much they matter.

In [ ]:
mapping_path = hs.write_mapping_doc(changes)
print(f"Wrote {mapping_path.relative_to(ROOT)}")

## 2. Build the household shock and run the checks

In [ ]:
out, _ = hs.build(master, cpi)
checks = hs.checks(out, master, cpi)
assert checks["result"].eq("pass").all(), checks[checks["result"] != "pass"]
out["all"] = "Rwanda"
w = out["weight"]
checks

In [ ]:
x = out["price_change_pct"]
q = {f"P{int(p*100)}": hs.weighted_quantile(x, w, p) for p in (0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95)}
summary = pd.Series({"weighted mean": np.average(x, weights=w), **q,
                     "annualised, weighted mean": np.average(out["annualised_inflation_pct"], weights=w),
                     "basket share of consumption (mean)": 100 * np.average(out["basket_share_of_consumption"], weights=w),
                     "extra annual cost of the same basket, RWF (median)": hs.weighted_quantile(out["extra_cost_same_basket_rwf"], w, 0.5),
                     "  per adult equivalent, RWF (median)": hs.weighted_quantile(out["extra_cost_same_basket_rwf_per_ae"], w, 0.5)})
summary.to_frame("value")

### Why the household measure is above published headline CPI

The bridge below splits the difference into three steps:
1. **Aggregation gap.** Published components weighted by the CPI's own (EICV4) weights do not exactly reproduce the published headline.
2. **Spending-share effect.** EICV7 households' own shares in place of the CPI weights, using only indices published for both areas.
3. **Urban-only index effect.** The main measure's use of the urban-only energy, fresh, local and imported indices.

In [ ]:
bridge = hs.headline_bridge(out, master, changes)
display(bridge)
wc = hs.weights_comparison(master)
print("Components whose EICV7 share differs most from the CPI weight (gap contribution, pp):")
display(wc.assign(abs_gap=wc["gap_contribution_pp"].abs()).sort_values(["area", "abs_gap"], ascending=[True, False])
        .groupby("area").head(6).drop(columns="abs_gap").set_index(["area", "component"]))

## 3. Category shocks

For each lens: the share of the basket, the price change of that part of the basket (household-specific where several indices apply), and the shock in percentage points of the total (share × change). Energy and transport overlap on vehicle fuel.

In [ ]:
rows = {}
for area, g in [("Rwanda", out), *out.groupby("area")]:
    gw = g["weight"]
    for lens in hs.GROUPS:
        pc = g[f"{lens}_price_change_pct"]
        rows[(lens, area)] = {"share of basket, %": 100 * np.average(g[f"{lens}_share_of_basket"], weights=gw),
                              "price change, % (households with spending)": np.average(pc[pc.notna()], weights=gw[pc.notna()]),
                              "shock, pp of total": np.average(g[f"{lens}_shock_pp"], weights=gw)}
pd.DataFrame(rows).T.unstack(1).round(2)

## 4. Charts

Each chart is followed by a one-line interpretation computed from the numbers. The comparison charts split the weighted mean price change into percentage-point contributions by spending group (the segments add up to the total). Tables with the same numbers and design-based standard errors follow in section 5.

In [ ]:
LIV = {"own_farming": "Own farming", "farm_wage": "Farm wage work", "nonfarm_wage": "Non-farm wage work",
       "nonfarm_self_employment": "Non-farm self-employment", "no_employed_member": "No employed member"}
interpret = {}

path = hs.plot_distribution(out, FIG / "shock_distribution.png")
display(Image(str(path)))
med = q["P50"]
top_seg = max(SEG, key=lambda s: np.average(out[f"contrib_{s}_pp"], weights=w))
interpret["shock_distribution"] = (
    f"The typical household's basket costs {med:.1f}% more than in January 2024 (80% of households between {q['P10']:.1f}% and {q['P90']:.1f}%), "
    f"above published headline CPI (urban {head['urban']:.1f}%, rural {head['rural']:.1f}%; see the bridge in section 2); "
    f"{hs.SEGMENTS[top_seg][0].lower()} contributes {np.average(out[f'contrib_{top_seg}_pp'], weights=w):.1f} pp of the average.")
display(Markdown(f"**Interpretation:** {interpret['shock_distribution']}"))

In [ ]:
t_area = pd.concat([hs.group_means(out, "all"), hs.group_means(out, "area", ["urban", "rural"], {"urban": "Urban", "rural": "Rural"})])
d_ur, se_ur = hs.domain_contrast(out["price_change_pct"], w, out["area"].eq("urban"), out["area"].eq("rural"), out["strata_id"], out["clust"])
title = {"almost the same": "Urban and rural baskets rose by almost the same, for different reasons",
         "a slightly larger": "Urban baskets rose slightly more than rural ones, for different reasons",
         "a slightly smaller": "Urban baskets rose slightly less than rural ones, for different reasons"}[hs.compare_phrase(d_ur, se_ur)]
path = hs.plot_decomposition(t_area, title,
                             "Weighted mean price change of the household's own basket, Jan 2024 to Sep 2026, by spending group", FIG / "shock_rural_urban.png")
display(Image(str(path)))
cd = {a: np.average(out.loc[out["area"].eq(a), "price_change_pct__common_detail"], weights=w[out["area"].eq(a)]) for a in ("urban", "rural")}
interpret["shock_rural_urban"] = (
    f"Urban {t_area.loc['Urban', 'mean']:.1f}% vs rural {t_area.loc['Rural', 'mean']:.1f}% ({d_ur:+.1f} pp, SE {se_ur:.2f}); contributions differ most for "
    f"{hs.explain_difference(t_area, 'Urban', 'Rural', 3)}; with only indices published for both areas, urban falls to "
    f"{cd['urban']:.1f}% (rural {cd['rural']:.1f}%), so the sign of the gap depends on the urban-only energy index.")
display(Markdown(f"**Interpretation:** {interpret['shock_rural_urban']}"))

In [ ]:
t_pov = hs.group_means(out, "poor", [1, 0], {1: "Poor", 0: "Non-poor"})
d, se = hs.domain_contrast(out["price_change_pct"], w, out["poor"].eq(1), out["poor"].eq(0), out["strata_id"], out["clust"])
title = {"almost the same": "Poor and non-poor face almost the same price rise; more of it is food",
         "a slightly larger": "Poor households face a slightly larger price rise than non-poor households",
         "a slightly smaller": "Poor households face a slightly smaller price rise than non-poor households"}[hs.compare_phrase(d, se)]
path = hs.plot_decomposition(t_pov, title, "Weighted mean price change of the household's own basket, Jan 2024 to Sep 2026, by spending group",
                             FIG / "shock_poor_nonpoor.png")
display(Image(str(path)))
interpret["shock_poor_nonpoor"] = (
    f"Poor {t_pov.loc['Poor', 'mean']:.1f}% vs non-poor {t_pov.loc['Non-poor', 'mean']:.1f}% ({d:+.2f} pp, SE {se:.2f}, "
    f"{'not statistically different' if abs(d) < 2 * se else 'statistically different'}); contributions differ most for "
    f"{hs.explain_difference(t_pov, 'Poor', 'Non-poor', 3)}, which largely offset each other.")
display(Markdown(f"**Interpretation:** {interpret['shock_poor_nonpoor']}"))

In [ ]:
t_prov = hs.group_means(out, "province_name").sort_values("mean", ascending=False)
t_prov.index = t_prov.index.str.replace(" Province", "")
path = hs.plot_decomposition(t_prov, "Average household price rise by province",
                             "Weighted mean price change of the household's own basket, Jan 2024 to Sep 2026, by spending group", FIG / "shock_by_province.png")
display(Image(str(path)))
hi, lo = t_prov.index[0], t_prov.index[-1]
interpret["shock_by_province"] = (
    f"{hi} has the highest average rise ({t_prov.loc[hi, 'mean']:.1f}%) and {lo} the lowest ({t_prov.loc[lo, 'mean']:.1f}%); "
    f"of the {t_prov.loc[hi, 'mean'] - t_prov.loc[lo, 'mean']:.1f} pp gap, contributions differ most for {hs.explain_difference(t_prov, hi, lo)}.")
display(Markdown(f"**Interpretation:** {interpret['shock_by_province']}"))

In [ ]:
t_liv = hs.group_means(out, "main_livelihood", list(LIV), LIV).sort_values("mean", ascending=False)
path = hs.plot_decomposition(t_liv, "Average household price rise by main income source",
                             "Main income source = livelihood of most employed members (EICV7 has no income aggregate)", FIG / "shock_by_income_source.png")
display(Image(str(path)))
hi, lo = t_liv.index[0], t_liv.index[-1]
interpret["shock_by_income_source"] = (
    f"The '{hi}' group faces the largest rise ({t_liv.loc[hi, 'mean']:.1f}%) and the '{lo}' group the smallest ({t_liv.loc[lo, 'mean']:.1f}%); "
    f"contributions differ most for {hs.explain_difference(t_liv, hi, lo)}.")
display(Markdown(f"**Interpretation:** {interpret['shock_by_income_source']}"))

In [ ]:
out["vup_status"] = np.where(out["vup_any"].eq(1), "Any VUP component", "No VUP")
parts = [hs.group_means(out, "vup_status", ["No VUP", "Any VUP component"])]
for col, lab in [("vup_direct_support", "– Direct Support"), ("vup_classic_pw", "– Classic Public Works"), ("vup_expanded_pw", "– Expanded Public Works"),
                 ("vup_nsds", "– NSDS"), ("vup_financial_services", "– Financial Services")]:
    parts.append(hs.group_means(out, col, [1], {1: lab}))
t_vup = pd.concat(parts)
d, se = hs.domain_contrast(out["price_change_pct"], w, out["vup_any"].eq(1), out["vup_any"].eq(0), out["strata_id"], out["clust"])
title = {"almost the same": "VUP participants and non-participants face almost the same price rise",
         "a slightly larger": "VUP participants face a slightly larger price rise than non-participants",
         "a slightly smaller": "VUP participants face a slightly smaller price rise than non-participants"}[hs.compare_phrase(d, se)]
path = hs.plot_decomposition(t_vup, title, "Weighted mean price change, Jan 2024 to Sep 2026. Rows marked – are components (146 households receive two or three)",
                             FIG / "shock_by_vup.png")
display(Image(str(path)))
interpret["shock_by_vup"] = (
    f"VUP participants face {t_vup.loc['Any VUP component', 'mean']:.1f}% vs {t_vup.loc['No VUP', 'mean']:.1f}% for non-participants "
    f"({d:+.2f} pp, SE {se:.2f}); contributions differ most for {hs.explain_difference(t_vup, 'Any VUP component', 'No VUP', 3)}.")
display(Markdown(f"**Interpretation:** {interpret['shock_by_vup']}"))

## 5. Group comparisons with standard errors

Weighted means (household weight) with design-based standard errors: Taylor linearisation with `strata_id` and `clust`, as in NISR's own tables. "Difference" is the first group minus the second.

In [ ]:
y = out["price_change_pct"]
comp_rows = []
for label, a, b in [("Poor vs non-poor", out["poor"].eq(1), out["poor"].eq(0)),
                    ("Rural vs urban", out["area"].eq("rural"), out["area"].eq("urban")),
                    ("VUP participants vs non-participants", out["vup_any"].eq(1), out["vup_any"].eq(0))]:
    ma, sa = hs.domain_mean(y, w, a, out["strata_id"], out["clust"])
    mb, sb = hs.domain_mean(y, w, b, out["strata_id"], out["clust"])
    dd, sd = hs.domain_contrast(y, w, a, b, out["strata_id"], out["clust"])
    comp_rows.append({"comparison": label, "first group, %": ma, "SE": sa, "second group, %": mb, "SE ": sb,
                      "difference, pp": dd, "SE of difference": sd, "|t|": abs(dd / sd), "households (first, second)": f"{int(a.sum()):,}, {int(b.sum()):,}"})
comparisons = pd.DataFrame(comp_rows).set_index("comparison")
comparisons

In [ ]:
seg_cols = [f"contrib_{s}_pp" for s in SEG]
display(pd.concat({"Area": t_area, "Poverty": t_pov, "Province": t_prov, "Main income source": t_liv, "VUP": t_vup})[["households", "mean", "se", "median", *seg_cols]])

## 6. Districts: top and bottom 10 by average shock

In [ ]:
t_dist = hs.group_means(out, "district_name")
t_dist["province"] = out.groupby("district_name")["province_name"].first().str.replace(" Province", "")
t_dist["% rural (weighted)"] = out.groupby("district_name").apply(lambda g: 100 * np.average(g["area"].eq("rural"), weights=g["weight"]))
t_dist["largest contributor"] = t_dist[seg_cols].idxmax(axis=1).str.replace("contrib_", "").str.replace("_pp", "")
t_dist = t_dist.sort_values("mean", ascending=False)
t_dist["rank"] = np.arange(1, len(t_dist) + 1)
cols = ["rank", "province", "households", "% rural (weighted)", "mean", "se", "median", "contrib_food_pp", "contrib_energy_pp", "contrib_communication_pp", "largest contributor"]
print("Top 10 districts (highest average price change)")
display(t_dist.head(10)[cols])
print("Bottom 10 districts (lowest average price change)")
display(t_dist.tail(10)[cols])
print(f"Range across districts: {t_dist['mean'].min():.1f}% to {t_dist['mean'].max():.1f}%")

## 7. Sensitivity of the shock to index choices

Each scenario changes one choice and keeps the rest. Official index values are never altered; scenarios only swap which official index prices a cell, or which cells enter the basket.

In [ ]:
scen_rows = []
for name, spec in hs.SCENARIOS.items():
    col = "price_change_pct" if name == "main" else f"price_change_pct__{name}"
    r = {"scenario": name, "description": spec["description"], "Rwanda": np.average(out[col], weights=w)}
    for a in ("urban", "rural"):
        m = out["area"].eq(a)
        r[a] = np.average(out.loc[m, col], weights=w[m])
    r["rural - urban"] = r["rural"] - r["urban"]
    r["poor - non-poor"] = (np.average(out.loc[out["poor"].eq(1), col], weights=w[out["poor"].eq(1)])
                            - np.average(out.loc[out["poor"].eq(0), col], weights=w[out["poor"].eq(0)]))
    r["rank correlation with main"] = out[col].rank().corr(out["price_change_pct"].rank())
    scen_rows.append(r)
scenarios = pd.DataFrame(scen_rows).set_index("scenario")
scenarios

## 8. Cost-of-Living Distress Score

A transparent index, not a model: six components, each scaled to 0–1 (higher means more distress), averaged with equal weights and multiplied by 100. Nothing is estimated or tuned.

In [ ]:
comp = pd.DataFrame({"definition": hs.DISTRESS_COMPONENTS})
comp["weighted mean"] = [np.average(out[c], weights=w) for c in comp.index]
for a in ("urban", "rural"):
    m = out["area"].eq(a)
    comp[f"mean, {a}"] = [np.average(out.loc[m, c], weights=w[m]) for c in comp.index]
comp

In [ ]:
qd = out.groupby("distress_quintile").apply(lambda g: pd.Series({
    "households": len(g), "score range": f"{g['distress_score'].min():.0f}-{g['distress_score'].max():.0f}",
    "mean price change, %": np.average(g["price_change_pct"], weights=g["weight"]),
    "poor, % of people": 100 * np.average(g["poor"], weights=g["pop_wt"]),
    "rural, % of households": 100 * np.average(g["area"].eq("rural"), weights=g["weight"]),
    "VUP, % of households": 100 * np.average(g["vup_any"], weights=g["weight"]),
    "food share (mean)": 100 * np.average(master.set_index("hhid").loc[g["hhid"], "share_food"], weights=g["weight"])}))
qd

In [ ]:
print("VUP coverage by need quintile (score without the VUP component, so coverage is not built into the ranking)")
cov = out.groupby("distress_quintile_excl_vup").apply(lambda g: pd.Series({
    "households": len(g), "poor, % of people": 100 * np.average(g["poor"], weights=g["pop_wt"]),
    "VUP, % of households": 100 * np.average(g["vup_any"], weights=g["weight"]),
    "weighted households": g["weight"].sum()}))
display(cov)
top = out["distress_quintile_excl_vup"].eq(5)
print(f"Households in the highest need quintile without any VUP component: {100 * np.average(out.loc[top, 'vup_any'].eq(0), weights=w[top]):.1f}% "
      f"(about {w[top & out['vup_any'].eq(0)].sum():,.0f} households)")
print("Robustness: leave each component out and compare rankings")
display(hs.leave_one_out(out.set_index("hhid"), w.set_axis(out["hhid"])))
ds = pd.concat({k: hs.group_means(out, *v, value="distress_score")[["households", "mean", "se"]] for k, v in {
    "Area": ("area", ["urban", "rural"]), "Poverty": ("poor", [1, 0], {1: "Poor", 0: "Non-poor"}),
    "VUP": ("vup_status", ["No VUP", "Any VUP component"]), "Main income source": ("main_livelihood", list(LIV), LIV)}.items()})
print("Mean distress score by group")
ds

## 9. Save

In [ ]:
final = out.drop(columns=["all", "vup_status"])
path = hs.save(final)
check = pd.read_parquet(path)
assert check["hhid"].is_unique and len(check) == 15_054
print(f"{path.relative_to(ROOT)}: {check.shape[0]:,} rows x {check.shape[1]} columns")
print("Figures:", ", ".join(sorted(p.name for p in FIG.glob("*.png"))))
for k, v in interpret.items():
    print(f"- {k}: {v}")